# Practical 12 homework — compressing a YOLO colony detector

The supplied compression script uses the validated exporter from `week7`. It creates a strict full-INT8 model with separate box and confidence outputs, which is the format used by the ESP32 firmware.

## Rules and deliverables

1. Optionally retrain your yolo model using provided(already labeled) data + if you wish data you labeled. If you will participate in esp32 practical train yolo11n, if you wouldnt train larger model of your liking.
2. Compress the model using provided script;
3. Check how accuracy will decrease after compression
4. a short report answering the questions at the end.

The board-compatible candidate is the `*_int8_split_224.tflite` file.

In [ ]:
from pathlib import Path
import json
import subprocess
import sys

from ultralytics import YOLO

WEEK12 = Path.cwd()
MODEL_PATH = WEEK12 / "student_model.pt"  # TODO: change to your checkpoint
DATA_YAML = WEEK12 / "data.yaml"  # TODO: point to your train/val YAML
IMG_SIZE = 224
OUTPUT_DIR = WEEK12 / "esp32_compression_artifacts"

assert MODEL_PATH.exists(), MODEL_PATH
assert DATA_YAML.exists(), DATA_YAML
print("Model:", MODEL_PATH)
print("Data:", DATA_YAML)

## 1. Optional retraining

If you retrain, use the same class definition as Practical 7 and record the image size, number of epochs, augmentations, and split strategy. A minimal command-line example is shown below; adapt it to your checkpoint and dataset.

In [ ]:
# Optional training example — run only if you want to retrain.
# model = YOLO(WEEK12 / ".." / "week7" / "best.pt")
# results = model.train(
#     data=str(DATA_YAML),
#     imgsz=IMG_SIZE,
#     epochs=30,
#     batch=16,
#     project=str(WEEK12 / "runs"),
#     name="student_yolo",
# )
# MODEL_PATH = WEEK12 / "runs" / "student_yolo" / "weights" / "best.pt"

## 2. Evaluate the original checkpoint

Keep this result as the reference. Use the same validation YAML and image size when comparing the compressed artifacts.

In [ ]:
model = YOLO(str(MODEL_PATH))
baseline_metrics = model.val(
    data=str(DATA_YAML),
    imgsz=IMG_SIZE,
    split="val",
    device="cpu",
    plots=False,
)
print("Original checkpoint evaluated. Record mAP50, mAP50-95, precision, and recall in the report.")

## 3. Convert the model for ESP32

The script creates FP16 and strict split-output INT8 artifacts. The representative calibration images should resemble the images that the model will receive.

In [ ]:
compression_script = WEEK12 / "compress_yolo_for_esp32.py"
command = [
    sys.executable, str(compression_script),
    "--model", str(MODEL_PATH),
    "--data", str(DATA_YAML),
    "--imgsz", str(IMG_SIZE),
    "--output-dir", str(OUTPUT_DIR),
]
# TODO: run this cell after checking the paths above.
# subprocess.run(command, check=True)

In [ ]:
# Inspect the generated artifacts.
for path in sorted(OUTPUT_DIR.glob("*224.*")):
    print(f"{path.name:45s} {path.stat().st_size / 1024**2:.3f} MiB")

summary_files = list(OUTPUT_DIR.glob("*_compression_summary.json"))
if summary_files:
    summary = json.loads(summary_files[0].read_text())
    print(json.dumps(summary, indent=2)[:4000])

## 4. Evaluate after compression

Start with the FP16 artifact. Then attempt the INT8 artifact. Depending on the installed Ultralytics/TFLite backend, the split-output INT8 model may require the custom decoder used in the week7 firmware rather than the standard `YOLO(...).val()` path. If that happens, report the error and use the provided local pipeline evaluator plus the board output.

In [ ]:
stem = MODEL_PATH.stem
fp16_path = OUTPUT_DIR / f"{stem}_fp16_{IMG_SIZE}.tflite"
int8_path = OUTPUT_DIR / f"{stem}_int8_split_{IMG_SIZE}.tflite"

for compressed_path in (fp16_path, int8_path):
    print("\nEvaluating:", compressed_path)
    if not compressed_path.exists():
        print("Missing artifact — run the conversion cell first.")
        continue
    try:
        compressed_model = YOLO(str(compressed_path))
        compressed_model.val(
            data=str(DATA_YAML), imgsz=IMG_SIZE, split="val",
            device="cpu", plots=False,
        )
    except Exception as exc:
        print(type(exc).__name__, exc)
        print("Use the week7 ESP32 pipeline decoder for split INT8 evaluation.")

## 5. Visual comparison

Choose several validation images, including easy, crowded, dim, blurry, and unusual examples. Compare the original and compressed predictions by drawing their boxes and confidence values.

In [ ]:
# TODO: select at least five validation images and complete this comparison.
# validation_image = Path("path/to/validation/image.jpg")
# baseline_result = model.predict(str(validation_image), imgsz=IMG_SIZE, conf=0.10, verbose=False)[0]
# compressed_result = YOLO(str(fp16_path)).predict(str(validation_image), imgsz=IMG_SIZE, conf=0.10, verbose=False)[0]
# display(Image.fromarray(baseline_result.plot()))
# display(Image.fromarray(compressed_result.plot()))

## Report questions

1. Did you retrain the Practical 7 model? If yes, what data did you add, how did you keep the split independent, and what changed in the training configuration?
2. Compare the original and compressed models using the available detection metrics. Report the metric, confidence threshold, IoU threshold, image size, and evaluation split.
3. How did FP16 and INT8 change file size, inference latency, and prediction confidence?
4. Show an images where compression preserved the predictions and three where the behaviour changed. Explain the likely cause of each change.
5. What do you expect to happen on realistic camera images that differ in illumination, focus, plate position, background, or colony density from the training data?
7. Which additional experiment would you run next: more representative calibration images, retraining with camera-like data, a different input size, confidence calibration, or a different model architecture? Explain why.